In [14]:
import os

import ee
#import geemap.foliumap as geemap
import geemap

from find_set_root import find_set_project_root
PROJECT_ROOT = find_set_project_root()
print(f"Project root found at: {PROJECT_ROOT}")
import utils.general_functions as ugf


DIR_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
DIR_DERIVED = os.path.join(PROJECT_ROOT, 'data', 'derived')
ugf.dir_ensure([DIR_RAW, DIR_DERIVED])

GEE_PROJECT = 'ee-tymc5571-multi-disturbance'

#Prepare to use Earth Engine
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
)
ee.Initialize(project=GEE_PROJECT)

Project root found at: C:\Users\tymc5571\dev\compound-disturbance-resilience
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\raw
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\derived


In [15]:
epa_lvl3 = ee.FeatureCollection("EPA/Ecoregions/2013/L3")

forest_masks = ee.ImageCollection("projects/ee-tymc5571-cbi-module/assets/forest_masks")
usfs = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/usfs_binary")

conservative_forest_mask = (
    forest_masks
    .select("conservative")
    .reduce(ee.Reducer.max())
    .gt(0)
    .unmask(0)
    .rename("conservative_forest")
    .setDefaultProjection(ee.Image(forest_masks.first()).projection())
)

western_stusps = ee.List(["WA","OR","CA","NV","ID","MT","WY","UT","CO","AZ"])
states = ee.FeatureCollection("TIGER/2018/States").filter(ee.Filter.inList("STUSPS", western_stusps))
west_geom = states.geometry().dissolve(1)

lvl3_west = epa_lvl3.filterBounds(west_geom)

In [16]:
map = geemap.Map()
map.addLayer(lvl3_west, {}, 'potential ecoregions')
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [17]:
scale = conservative_forest_mask.projection().nominalScale()

# Area bands (m^2)
pixel_area = ee.Image.pixelArea()
forest_area_band = pixel_area.updateMask(conservative_forest_mask.eq(1)).rename("a_forest")
both_area_band = pixel_area.updateMask(conservative_forest_mask.eq(1).And(usfs.eq(1))).rename("a_both")
total_area_band = pixel_area.rename("a_total")

area_stack = ee.Image.cat([total_area_band, forest_area_band, both_area_band])

def add_stats(feat):
    feat = ee.Feature(feat)
    geom_int = feat.geometry().intersection(west_geom, ee.ErrorMargin(1))

    sums = area_stack.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom_int,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True,
        tileScale=8
    )

    a_total = ee.Number(sums.get("a_total"))
    a_forest = ee.Number(sums.get("a_forest"))
    a_both = ee.Number(sums.get("a_both"))

    pct_forest = ee.Algorithms.If(a_total.gt(0), a_forest.divide(a_total).multiply(100), None)
    pct_both = ee.Algorithms.If(a_total.gt(0), a_both.divide(a_total).multiply(100), None)

    return feat.set({
        "area_km2_total_in_west": a_total.divide(1e6),
        "area_km2_conservative_forest": a_forest.divide(1e6),
        "area_km2_both": a_both.divide(1e6),
        "pct_area_conservative_forest": pct_forest,
        "pct_area_both": pct_both
    })

table_fc = lvl3_west.map(add_stats)

In [18]:
task = ee.batch.Export.table.toDrive(
    collection=table_fc,
    description="L3_western_conservative_forest_usfs",
    folder="GEE_export_ecoregion_stats",
    fileFormat="CSV"
)
task.start()